In [1]:
%pip install --upgrade git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-pd3h13yl
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-pd3h13yl
  Resolved https://github.com/huggingface/transformers.git to commit f397b9e651dc7387de6bce551895619dfb1ec4f0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 11.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.8 MB/s eta 0:00:00:00:01
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11772970 sha256=e799a7afa55616bf75a8f6f1e6f951494ea9aca012bd8e87bbcbc92f42c12677
  Stored in directory: /tmp/pip-ephem-wheel-cache-i9xaasyy/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet
   

In [ ]:
import os
os._exit(0)

In [5]:
# Cell 2: 加载Gemma 4模型
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [6]:
import numpy as np
import timm
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
from shapely.geometry import Point
from geopy.distance import geodesic
import requests
import torch

class OilSpillDecisionTool:
    def __init__(self, coastline_path, ais_path):
        self.coastline_gdf = gpd.read_file(coastline_path)
        if self.coastline_gdf.crs is None:
            self.coastline_gdf = self.coastline_gdf.set_crs("EPSG:4326", allow_override=True)
        elif self.coastline_gdf.crs.to_string() != "EPSG:4326":
            self.coastline_gdf = self.coastline_gdf.to_crs("EPSG:4326")
        self.ais_df = pd.read_csv(ais_path)
        self.ais_df['timestamp'] = pd.to_datetime(self.ais_df['timestamp'])

    def calculate_distance_to_coast(self, lon, lat):
        """精确计算点到海岸线的最短距离（海里）"""
        point = Point(lon, lat)
        coastline_union = self.coastline_gdf.geometry.union_all()
        nearest_point = coastline_union.interpolate(coastline_union.project(point))
        dist_km = geodesic((lat, lon), (nearest_point.y, nearest_point.x)).kilometers
        return dist_km / 1.852

    def find_recent_vessels(self, lon, lat, incident_time, hours=6, radius_nm=2):
        """
        查找 incident_time 前 hours 小时内，经过 (lon,lat) 方圆 radius_nm 海里的船只。
        返回 MMSI 列表。
        """
        start_time = pd.to_datetime(incident_time) - timedelta(hours=hours)
        mask_time = (self.ais_df['timestamp'] >= start_time) & (self.ais_df['timestamp'] <= incident_time)
        df_time = self.ais_df[mask_time]
        if df_time.empty:
            return []
        # 距离阈值：海里转公里（1海里=1.852公里）
        radius_km = radius_nm * 1.852
        distances = df_time.apply(lambda row: geodesic((row['latitude'], row['longitude']), (lat, lon)).km, axis=1)
        nearby = df_time[distances <= radius_km]
        return nearby['mmsi'].unique().tolist()

    def get_decision_chain(self, lon, lat, incident_time):
        """
        返回结构化信息，包含距离、是否<=20海里、船只列表。
        """
        distance_nm = self.calculate_distance_to_coast(lon, lat)
        within_20 = distance_nm <= 20.0
        vessels = []
        if within_20:
            vessels = self.find_recent_vessels(lon, lat, incident_time, hours=6, radius_nm=2)
        return {
            "lon": lon,
            "lat": lat,
            "distance_nm": distance_nm,
            "within_20": within_20,
            "vessels": vessels
        }

# 初始化 
coastline_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/coastline/ne_10m_coastline.shp"
ais_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/ais/sample_ais.csv"
model_path = "/kaggle/input/datasets/qihuiren/oil-detect-dataset/kaggle_dataset/best_student.pth"

tool = OilSpillDecisionTool(coastline_path, ais_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classification_model = timm.create_model("mobilenetv3_small_100", num_classes=2, pretrained=False);
state_dict = torch.load(model_path, map_location=device)
classification_model.load_state_dict(state_dict)
classification_model = classification_model.to(device)
classification_model.eval()

MobileNetV3(
  (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): Hardswish()
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
        (bn1): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): ReLU(inplace=True)
          (conv_expand): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (gate): Hardsigmoid()
        )
        (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): B

In [30]:
def generate_report_with_gemma(has_oil, decision_info, image_filename):
    lon = decision_info["lon"]
    lat = decision_info["lat"]
    dist = decision_info["distance_nm"]
    within_20 = decision_info["within_20"]
    vessels = decision_info["vessels"]

    # 无泄漏：简单返回
    if not has_oil:
        return f"该坐标[{lon:.4f}, {lat:.4f}]无泄漏"
        
    # 固定前缀
    coord_prefix = f"该坐标[{lon:.4f}, {lat:.4f}]"
    
    # 有泄漏：让 Gemma 生成报告
    if not within_20:
        data_desc = f"距离海岸线{dist:.2f}海里，大于20海里，安全"
        fallback_report = f"{coord_prefix}有泄漏，距离海岸线{dist:.2f}海里，大于20海里，安全"
    else:
        if vessels:
            vessel_str = "、".join(str(v) for v in vessels)
            data_desc = f"距离海岸线{dist:.2f}海里，小于20海里，可能的泄露船只编号：[{vessel_str}]"
            fallback_report = f"{coord_prefix}有泄漏，距离海岸线{dist:.2f}海里，小于20海里，可能的泄露船只编号：[{vessel_str}]"
        else:
            data_desc = f"距离海岸线{dist:.2f}海里，小于20海里，未找到可能泄露船只"
            fallback_report = f"{coord_prefix}有泄漏，距离海岸线{dist:.2f}海里，小于20海里，未找到可能泄露船只"
    
    prompt = f"""你是一个海洋油污应急响应助手。请根据以下信息，生成一句简短的中文描述，不要包含坐标信息（坐标已经由系统提供）。只输出描述内容。
    
    示例：
    信息：距离海岸线15.3海里，小于20海里，可能的泄露船只编号：[123456789]
    输出：有泄漏，距离海岸线15.3海里，小于20海里，可能的泄露船只编号：[123456789]
    
    {data_desc}"""

    # 调用 Gemma4
    inputs = tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
    input_len = inputs['input_ids'].shape[1]
    outputs = gemma_model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.3,          # 较低温度，保持稳定
        do_sample=True,
        repetition_penalty=1.1,
        top_p=0.9,
    )
    new_tokens = outputs[0][input_len:]
    gemma_desc = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # 如果模型输出异常（包含示例、方括号、冒号等），使用 fallback
    if not gemma_desc or "示例" in gemma_desc or gemma_desc.startswith("[") or "：" in gemma_desc[:10]:
        final_report = fallback_report
    else:
        # 正常情况：拼接坐标前缀和模型生成的描述
        # 注意：模型输出可能已经包含了“有泄漏”，为避免重复，可以清理一下
        if gemma_desc.startswith("有泄漏"):
            # 如果描述已经以“有泄漏”开头，直接拼接
            final_report = f"{coord_prefix}{gemma_desc}"
        else:
            # 否则加上“有泄漏”，但一般模型输出会包含“有泄漏”
            final_report = f"{coord_prefix}有泄漏，{gemma_desc}"

    return final_report

In [31]:
# Cell 4: 完整测试流程
from datetime import datetime
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# 定义验证集预处理（与训练时一致）
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_oil(image_path, model, transform, device, threshold=0.5):
    img = Image.open(image_path).convert('L')
    img = img.convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1)
        oil_prob = probs[0, 1].item()
    return oil_prob > threshold, oil_prob

# ---------- 测试数据 ----------
test_cases = [
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_dtsR0emPVPPNdxSS_SFr_cls_1.jpg",
        "lon": -89.382194,
        "lat": 28.801274,
        "timestamp": datetime(2024, 1, 9, 3, 0, 0),
        "filename": "test1.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -89.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test2.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/0_0_0_img_5kPtgTDfaBqbQJtE_ADR_cls_0.jpg",
        "lon": -92.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test3.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -92.814811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test4.jpg"
    },
    {
        "image_path": "/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data/Samples/1_200_0_img_9c3d5585_TRI_cls_1.jpg",
        "lon": -89.614811,
        "lat": 29.276685,
        "timestamp": datetime(2024, 1, 5, 1, 0, 0),
        "filename": "test5.jpg"
    },
]

for case in test_cases:
    has_oil, prob = predict_oil(
        case["image_path"],
        classification_model,
        val_transform,
        device,
        threshold=0.5
    )

    if not has_oil:
        # 无泄漏，直接生成无泄漏报告
        decision_info = tool.get_decision_chain(case["lon"], case["lat"], case["timestamp"])
        report = generate_report_with_gemma(has_oil, decision_info, case["filename"])
        print(f"=== 报告 {case['filename']} ===\n{report}\n")
    else:
        # 有泄漏，获取决策信息（距离、是否<=20、船只列表）
        decision_info = tool.get_decision_chain(case["lon"], case["lat"], case["timestamp"])
        report = generate_report_with_gemma(has_oil, decision_info, case["filename"])
        print(f"=== 报告 {case['filename']} ===\n{report}\n")
        # 可选：保存到文件
        with open(f"{case['filename']}_report.txt", "w") as f:
            f.write(report)

=== 报告 test1.jpg ===
该坐标[-89.3822, 28.8013]有泄漏，距离海岸线7.78海里，小于20海里，可能的泄露船只编号：[308457070]

=== 报告 test2.jpg ===
该坐标[-89.8148, 29.2767]有泄漏，距离海岸线2.30海里，小于20海里，可能的泄露船只编号：[816035178]

=== 报告 test3.jpg ===
该坐标[-92.8148, 29.2767]无泄漏

=== 报告 test4.jpg ===
该坐标[-92.8148, 29.2767]有泄漏，距离海岸线20.83海里，大于20海里，安全

=== 报告 test5.jpg ===
该坐标[-89.6148, 29.2767]有泄漏，距离海岸线0.24海里，小于20海里，未找到可能泄露船只

